# Stress test 4 — chunking & the session read cache

Sub-file dedup efficiency, incompressible data, and the read-cache lifecycle (node dirs stay packed; reads route through `.cache`).

In [1]:
import shutil
import os
import numpy as np
import pandas as pd
import ancestree
from ancestree import LineageStore
from ancestree.chunkstore import ChunkStore

SCRATCH = "_stress"  # all stores written here; safe to delete
shutil.rmtree(SCRATCH, ignore_errors=True)
os.makedirs(SCRATCH, exist_ok=True)


def store(name, **kw):
    return LineageStore(f"{SCRATCH}/{name}", **kw)


def node_dirs(s):
    return [d for d in os.listdir(s.root) if not d.startswith(".")]


print("ancestree", getattr(ancestree, "__version__", "?"))

ancestree 0.1.0


## Sub-file sharing: many near-identical large CSVs

In [2]:
s = store("near", chunk=True)
rng = np.random.default_rng(0)
base = pd.DataFrame({"x": np.arange(80_000), "y": rng.normal(size=80_000).round(4)})
with s.create_node(step_type="ingest") as ing:
    base.to_csv(ing / "data.csv", index=False)
for i in range(8):  # 8 versions, each a tiny edit
    v = base.copy()
    v.loc[i * 1000, "y"] = 999.0
    with s.create_node(step_type="clean", parent=ing) as c:
        v.to_csv(c / "data.csv", index=False)

cs = ChunkStore(s.root)
pool = sum(cs._path(d).stat().st_size for d in cs.all_digests())
one_csv = len(base.to_csv(index=False).encode())
print(
    f"9 ~identical CSVs | storing whole: {9 * one_csv / 1e6:.1f} MB | pool: {pool / 1e6:.2f} MB"
)

9 ~identical CSVs | storing whole: 9.5 MB | pool: 0.55 MB


## Incompressible data — chunking can't help (expected)

In [3]:
s2 = store("random", chunk=True)
for i in range(3):
    with s2.create_node(step_type="blob") as n:
        (n / "r.bin").write_bytes(os.urandom(500_000))
cs2 = ChunkStore(s2.root)
pool2 = sum(cs2._path(d).stat().st_size for d in cs2.all_digests())
print(f"3 x 500KB random: pool {pool2 / 1e6:.2f} MB (~1.5 MB expected, no sharing)")

3 x 500KB random: pool 1.50 MB (~1.5 MB expected, no sharing)


## The read cache — node dirs stay packed; the path is node-scoped

In [4]:
s3 = store("cache", chunk=True)
with s3.create_node(step_type="x") as n:
    (n / "results/clean.csv").write_text("a,b\n1,2\n")
print("node dir after write:", sorted(os.listdir(f"{s3.root}/{n.node_id}")))

p = s3.get_node(n.node_id) / "results/clean.csv"  # transparent read
print("returned path:", str(p).split("_stress/")[-1])
print("read back:", repr(p.read_text()))
print(
    "artifact still NOT loose in node dir:",
    not os.path.exists(f"{s3.root}/{n.node_id}/results/clean.csv"),
)

node dir after write: ['.artifacts.json', 'meta.json', 'results']
returned path: cache/.cache/10988-4337783e/de23768d/results/clean.csv
read back: 'a,b\n1,2\n'
artifact still NOT loose in node dir: True


In [5]:
# Cache hit within a session, then clear -> regenerates from the pool
p2 = s3.get_node(n.node_id) / "results/clean.csv"
print("same cached path reused:", p == p2)
s3.clear_cache()
print(
    "after clear_cache, still readable:",
    (s3.get_node(n.node_id) / "results/clean.csv").read_text() == "a,b\n1,2\n",
)

same cached path reused: True
after clear_cache, still readable: True


**Cross-session note:** the cache is wiped when the kernel exits, so the *first* read in a new session re-decompresses from the pool (the node dir is never re-populated). Re-reads within a session are free.

## Edge cases that work correctly

In [6]:
s4 = store("edges", chunk=True)
with s4.create_node(step_type="x") as n:
    (n / "empty.bin").write_bytes(b"")  # 0-byte file
    (n / "deep/a/b/ünïcödé_🔥.txt").write_text("ok")  # nested + unicode
    (n / "dup.txt").write_text("v1")
    (n / "dup.txt").write_text("v2")  # overwrite
r = s4.get_node(n.node_id)
print("empty:", (r / "empty.bin").read_bytes() == b"")
print("nested+unicode:", (r / "deep/a/b/ünïcödé_🔥.txt").read_text() == "ok")
print("overwrite kept last:", (r / "dup.txt").read_text() == "v2")

empty: True
nested+unicode: True
overwrite kept last: True
